# GNN Baseline Comparison: GCN / GIN vs GAT

| Model | Aggregation | Edge Features | Parameters |
|-------|------------|---------------|------------|
| GCN | Mean (degree-norm) | 6d SMILES bonds | ~1.1M |
| GIN | Sum + MLP (\(\varepsilon=0\)) | 6d SMILES bonds | ~1.3M |
| GAT (ref) | 12-head attention | 23d (6 bond + 16 RBF + 1 BO) | ~1.3M |

All models share: 5 layers, 384d hidden, LayerNorm+residual, metal-centric pooling.
Same 95,152 train/val/test split as ensemble CV for fair comparison.
Figures use English labels.


In [1]:
import numpy as np, pandas as pd, time, gzip, os, sys, json
import matplotlib.pyplot as plt, warnings
from collections import Counter
warnings.filterwarnings("ignore")

import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.amp import autocast, GradScaler

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    p = torch.cuda.get_device_properties(0)
    print(f"VRAM: {p.total_memory/1024**3:.1f} GB")

from sklearn.metrics import mean_absolute_error, r2_score
from tqdm import tqdm

df_y = pd.read_csv("tmQM_y.csv", sep=";")
df_y = df_y.rename(columns={
    "HL_Gap": "gap", "HOMO_Energy": "homo", "LUMO_Energy": "lumo",
    "Dipole_M": "dipole", "Metal_q": "metal_charge",
    "Electronic_E": "E_elec", "Dispersion_E": "E_disp",
    "Polarizability": "polarizability"
})
targets = ["gap","homo","lumo","dipole","metal_charge","E_elec","E_disp","polarizability"]
print(f"Loaded {len(df_y)} complexes")


Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.5 GB


Loaded 108541 complexes


In [2]:
print("="*60)
print("Data loading + SMILES graph building (6d edge, 8d node, no 3D features)")
print("="*60)

from rdkit import Chem
from rdkit import RDLogger; RDLogger.logger().setLevel(RDLogger.ERROR)

def atom_to_z(sym):
    tbl = {"H":1,"He":2,"Li":3,"Be":4,"B":5,"C":6,"N":7,"O":8,"F":9,"Ne":10,
           "Na":11,"Mg":12,"Al":13,"Si":14,"P":15,"S":16,"Cl":17,"Ar":18,
           "K":19,"Ca":20,"Sc":21,"Ti":22,"V":23,"Cr":24,"Mn":25,"Fe":26,
           "Co":27,"Ni":28,"Cu":29,"Zn":30,"Br":35,"Ru":44,"Rh":45,
           "Pd":46,"Ag":47,"Cd":48,"I":53,"Pt":78,"Au":79,"Hg":80,
           "Mo":42,"W":74,"Ir":77,"Os":76,"Re":75}
    return tbl.get(sym,0)

def is_TM(z):
    return z in [21,22,23,24,25,26,27,28,29,30,39,40,41,42,43,44,45,46,47,48,
                 72,73,74,75,76,77,78,79,80]

def parse_xyz(xyz_p, start_idx=0):
    mols = []
    with gzip.open(xyz_p, "rt") as f:
        blocks = f.read().strip().split(chr(10))
    i, mol_idx = 0, start_idx - 1
    while i < len(blocks):
        line = blocks[i].strip(); i += 1
        if not line: continue
        parts = line.split()
        if len(parts)==1 and parts[0].isdigit():
            n_atoms = int(parts[0])
            if i >= len(blocks): break
            i += 1
            atoms = []
            for _ in range(n_atoms):
                if i >= len(blocks): break
                cline = blocks[i].strip(); i += 1
                cparts = cline.split()
                if len(cparts)>=4:
                    try: atoms.append(atom_to_z(cparts[0]))
                    except ValueError: continue
            mol_idx += 1
            if any(is_TM(z) for z in atoms) and len(atoms)>1:
                mols.append(atoms)
    return mols

print("Parsing XYZ for atom lists...")
all_atoms = []
for part in [1,2,3]:
    atoms = parse_xyz(f"tmQM_X{part}.xyz.gz", start_idx=len(all_atoms))
    all_atoms.extend(atoms)
print(f"  {len(all_atoms)} TM complexes")

ELEMENT_DATA_SM = {1:(1,1),5:(2,13),6:(2,14),7:(2,15),8:(2,16),9:(2,17),14:(3,14),15:(3,15),16:(3,16),17:(3,17),20:(4,2),21:(4,3),22:(4,4),23:(4,5),24:(4,6),25:(4,7),26:(4,8),27:(4,9),28:(4,10),29:(4,11),30:(4,12),35:(4,17),42:(5,6),44:(5,8),45:(5,9),46:(5,10),47:(5,11),48:(5,12),53:(5,17),72:(6,4),73:(6,5),74:(6,6),75:(6,7),76:(6,8),77:(6,9),78:(6,10),79:(6,11),80:(6,12)}
BOND_TYPES_SM = [1,2,3,12]; B2I_SM = {b:i for i,b in enumerate(BOND_TYPES_SM)}; N_BT_SM = 4
smiles_by_idx = df_y["SMILES"].tolist()
csv_labels = df_y[targets].values.astype(np.float32)

def atom_features_sm(atom):
    z = atom.GetAtomicNum(); p,g = ELEMENT_DATA_SM.get(z,(0,0))
    return [z, atom.GetDegree(), atom.GetFormalCharge(), p, g,
            int(atom.GetIsAromatic()), atom.GetTotalNumHs(), int(z >= 21)]

print("Building SMILES-only graphs...")
graphs, y_vals, atoms_filt, failed = [], [], [], 0

for idx in tqdm(range(len(all_atoms)), desc="Graphs"):
    smi = smiles_by_idx[idx] if idx < len(smiles_by_idx) else ""
    if not isinstance(smi, str) or smi == "":
        failed += 1; continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None: failed += 1; continue
    N = mol.GetNumAtoms()
    xf = [atom_features_sm(a) for a in mol.GetAtoms()]
    x = torch.tensor(xf, dtype=torch.float32)
    src, dst, battr = [], [], []
    sm_z = [xf[i][0] for i in range(N)]
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = int(bond.GetBondType())
        za, zb = sm_z[i], sm_z[j]
        is_ml = float((za>=21)!=(zb>=21))
        is_ring = float(bond.IsInRing())
        src.extend([i,j]); dst.extend([j,i])
        bf = [0.0]*N_BT_SM
        if bt in B2I_SM: bf[B2I_SM[bt]] = 1.0
        bf.extend([is_ring, is_ml])
        battr.extend([bf,bf])
    ei = torch.tensor([src,dst], dtype=torch.long) if src else torch.zeros(2,0,dtype=torch.long)
    ea = torch.tensor(battr, dtype=torch.float32) if battr else torch.zeros(0,6)
    graphs.append({"x":x, "edge_index":ei, "edge_attr":ea, "num_nodes":N})
    y_vals.append(csv_labels[idx])
    atoms_filt.append(all_atoms[idx])

y_all = torch.tensor(y_vals, dtype=torch.float32)
mask = ~(torch.isnan(y_all).any(dim=1) | torch.isinf(y_all).any(dim=1))
graphs = [g for i,g in enumerate(graphs) if mask[i]]
y_all = y_all[mask]
atoms_filt = [a for i,a in enumerate(atoms_filt) if mask[i]]
print(f"Built {len(graphs):,} graphs ({failed} failed)")
print(f"Node dim: {graphs[0]["x"].shape[1]}, Edge dim: {graphs[0]["edge_attr"].shape[1]}")


Data loading + SMILES graph building (6d edge, 8d node, no 3D features)
Parsing XYZ for atom lists...


  102077 TM complexes
Building SMILES-only graphs...


Graphs:   0%|          | 0/102077 [00:00<?, ?it/s]

Graphs:   0%|          | 198/102077 [00:00<00:51, 1967.27it/s]

Graphs:   0%|          | 421/102077 [00:00<00:48, 2116.93it/s]

...

Graphs: 100%|█████████▉| 101746/102077 [01:09<00:00, 1402.83it/s]

Graphs: 100%|█████████▉| 101887/102077 [01:10<00:00, 1396.83it/s]

Graphs: 100%|█████████▉| 102031/102077 [01:10<00:00, 1409.44it/s]

Graphs: 100%|██████████| 102077/102077 [01:10<00:00, 1454.41it/s]

Built 94,598 graphs (7479 failed)
Node dim: 8, Edge dim: 6


In [3]:
print("="*60)
print("Data split (reuse ensemble CV split for fair comparison)")
print("="*60)

if os.path.exists("ensemble_split.npz"):
    split = np.load("ensemble_split.npz")
    te_idx = split["te_idx"]
    tr_idx = split["fold_0_train"]
    va_idx = split["fold_0_val"]
    print(f"Using ensemble_split.npz")
else:
    idx = np.random.RandomState(42).permutation(len(graphs))
    n_tr = int(len(graphs)*0.7)
    n_va = int(len(graphs)*0.15)
    tr_idx = idx[:n_tr]
    va_idx = idx[n_tr:n_tr+n_va]
    te_idx = idx[n_tr+n_va:]
print(f"Train: {len(tr_idx):,}  Val: {len(va_idx):,}  Test: {len(te_idx):,}")

# Pre-index for fast collation
g_x = [g["x"] for g in graphs]
g_ei = [g["edge_index"] for g in graphs]

y_all_gpu = y_all.to(device)
y_tr = y_all_gpu[tr_idx]
y_mean_local = y_tr.mean(dim=0)
y_std_local = y_tr.std(dim=0).clamp(min=1e-8)
y_norm = (y_all_gpu - y_mean_local) / y_std_local

transfer_stream = torch.cuda.Stream()

def collate_fast(indices):
    x_list = [g_x[i] for i in indices]
    ei_list = [g_ei[i] for i in indices]
    batch_list = []
    offset = 0
    for k, x in enumerate(x_list):
        n = x.size(0)
        batch_list.append(torch.full((n,), k, dtype=torch.long))
        ei_list[k] = ei_list[k] + offset
        offset += n
    with torch.cuda.stream(transfer_stream):
        batched = {
            "x": torch.cat(x_list, dim=0).to(device, non_blocking=True),
            "edge_index": torch.cat(ei_list, dim=1).to(device, non_blocking=True),
        }
        batch_idx = torch.cat(batch_list, dim=0).to(device, non_blocking=True)
    return batched, batch_idx

print("Ready.")


Data split (reuse ensemble CV split for fair comparison)
Using ensemble_split.npz
Train: 51,180  Val: 25,590  Test: 13,547


Ready.


In [4]:
print("="*60)
print("Model definitions: GCN + GIN")
print("="*60)

HIDDEN = 384; N_LAYERS = 5; DROPOUT = 0.10; N_TARGETS = 8

def scatter_mean_per_graph(h, batch, n):
    out = torch.zeros(n, h.size(1), device=h.device, dtype=h.dtype)
    out.scatter_add_(0, batch.unsqueeze(-1).expand_as(h), h)
    cnt = torch.zeros(n, device=h.device, dtype=h.dtype)
    cnt.scatter_add_(0, batch, torch.ones_like(batch, dtype=h.dtype))
    return out / cnt.clamp(min=1).unsqueeze(-1)

def scatter_max_per_graph(h, batch, n):
    out = torch.full((n, h.size(1)), float("-inf"), device=h.device, dtype=h.dtype)
    out.scatter_reduce_(0, batch.unsqueeze(-1).expand_as(h), h, reduce="amax", include_self=False)
    return out

class GCNConv(nn.Module):
    """Graph Convolution: h_i' = W * mean(h_j for j in N(i))"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, x, ei):
        src, dst = ei
        N = x.size(0)
        msg = x[src]
        aggr = torch.zeros(N, x.size(1), device=x.device, dtype=x.dtype)
        aggr.scatter_add_(0, dst.unsqueeze(-1).expand_as(msg), msg)
        deg = torch.zeros(N, device=x.device, dtype=x.dtype)
        deg.scatter_add_(0, dst, torch.ones_like(src, dtype=x.dtype))
        deg = deg.clamp(min=1)
        return self.lin(aggr / deg.unsqueeze(-1))

class GINConv(nn.Module):
    """Graph Isomorphism Network: h_i' = MLP((1+eps)*h_i + sum(h_j))"""
    def __init__(self, in_dim, out_dim, eps=0.0):
        super().__init__()
        self.eps = eps
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )
    def forward(self, x, ei):
        src, dst = ei
        N = x.size(0)
        aggr = torch.zeros(N, x.size(1), device=x.device, dtype=x.dtype)
        aggr.scatter_add_(0, dst.unsqueeze(-1).expand_as(x[src]), x[src])
        return self.mlp((1 + self.eps) * x + aggr)

class BaselineGNN(nn.Module):
    """Generic GNN backbone: GCN or GIN, same readout as GAT model"""
    def __init__(self, conv_type="gcn", node_dim=8, hidden=HIDDEN,
                 n_layers=N_LAYERS, output_dim=N_TARGETS, dropout=DROPOUT):
        super().__init__()
        self.conv_type = conv_type
        self.embed = nn.Linear(node_dim, hidden)
        Conv = GCNConv if conv_type == "gcn" else GINConv
        self.convs = nn.ModuleList([Conv(hidden, hidden) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.readout = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden//2, output_dim))

    def forward(self, g, batch=None):
        x, ei = g["x"], g["edge_index"]
        h = self.embed(x)
        for conv, norm in zip(self.convs, self.norms):
            h = norm(h + conv(h, ei))
            h = self.drop(h)
        if batch is None:
            mm = (x[:,7] > 0.5)
            mf = h[mm].mean(dim=0) if mm.any() else torch.zeros(h.size(1), device=h.device, dtype=h.dtype)
            return self.readout(torch.cat([h.mean(0), h.max(0).values, mf])).unsqueeze(0)
        n = int(batch.max().item()) + 1
        mean_h = scatter_mean_per_graph(h, batch, n)
        max_h = scatter_max_per_graph(h, batch, n)
        mm = (x[:,7] > 0.5)
        mf = scatter_mean_per_graph(h[mm], batch[mm], n) if mm.any() else torch.zeros(n, h.size(1), device=h.device, dtype=h.dtype)
        return self.readout(torch.cat([mean_h, max_h, mf], dim=1))

for ct in ["gcn", "gin"]:
    m = BaselineGNN(ct)
    print(f"{ct.upper()}: {sum(p.numel() for p in m.parameters()):,} params")


Model definitions: GCN + GIN
GCN: 1,262,792 params
GIN: 2,003,912 params


In [5]:
print("="*60)
print("Training function (prefetch + compile + bs=512)")
print("="*60)

def train_baseline(conv_type, seed=42):
    """Train one baseline model, return test predictions + metrics"""
    torch.manual_seed(seed); np.random.seed(seed)
    model = BaselineGNN(conv_type).to(device)
    try:
        model = torch.compile(model, mode="reduce-overhead")
        compiled = True
    except Exception:
        compiled = False
    opt = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2, eta_min=1e-6)
    scaler = GradScaler("cuda")

    bs = 512; best_va = float("inf"); best_state = None; patience = 0
    epochs = 200; early_stop = 40
    n_tr = len(tr_idx); n_va = len(va_idx)
    print(f"  [{conv_type.upper()}] epochs={epochs} bs={bs} compile={compiled}")

    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_tr)
        first_pos = perm[:bs]
        first_bidx = tr_idx[first_pos].tolist()
        next_batch = collate_fast(first_bidx)
        next_by = y_norm[first_bidx]

        for s in range(0, n_tr, bs):
            batched, batch_idx = next_batch
            by = next_by
            ns = s + bs
            if ns < n_tr:
                npos = perm[ns:ns+bs]
                nbidx = tr_idx[npos].tolist()
                next_batch = collate_fast(nbidx)
                next_by = y_norm[nbidx]
            torch.cuda.current_stream().wait_stream(transfer_stream)
            opt.zero_grad()
            with autocast("cuda"):
                loss = nn.MSELoss()(model(batched, batch_idx), by)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()

        sched.step()
        model.eval()
        with torch.no_grad():
            vp = []
            for s in range(0, n_va, bs):
                bidx = va_idx[s:s+bs].tolist()
                batched, batch_idx = collate_fast(bidx)
                with autocast("cuda"):
                    vp.append(model(batched, batch_idx).float())
            va_loss = nn.MSELoss()(torch.cat(vp), y_norm[va_idx])
        if va_loss < best_va:
            best_va = va_loss.item()
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
        if patience >= early_stop:
            print(f"  [{conv_type.upper()}] Early stop @ ep {ep+1}")
            break
        if (ep+1) % 40 == 0:
            print(f"  [{conv_type.upper()}] Ep {ep+1}: va={va_loss.item():.4f} pat={patience}")

    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        tp = []
        for s in range(0, len(te_idx), bs):
            bidx = te_idx[s:s+bs].tolist()
            batched, batch_idx = collate_fast(bidx)
            with autocast("cuda"):
                tp.append(model(batched, batch_idx).float().cpu())
        tp = torch.cat(tp)
    tp_orig = tp * y_std_local.cpu() + y_mean_local.cpu()
    tt = y_all[te_idx]
    r2s = [r2_score(tt[:,i].numpy(), tp_orig[:,i].numpy()) for i in range(N_TARGETS)]
    maes = [mean_absolute_error(tt[:,i].numpy(), tp_orig[:,i].numpy()) for i in range(N_TARGETS)]
    mean_r2 = np.mean(r2s)
    print(f"  [{conv_type.upper()}] Mean R2={mean_r2:.4f}  gap={r2s[0]:.4f}  dipole={r2s[3]:.4f}")
    return {"r2": r2s, "mae": maes, "mean_r2": mean_r2, "preds": tp_orig.numpy()}


Training function (prefetch + compile + bs=512)


In [6]:
print("="*60)
print("Running GCN + GIN baselines (bs=512, torch.compile)")
print("="*60)

results = {}
for ct in ["gcn", "gin"]:
    print(f"\n{"="*50}")
    print(f"{ct.upper()} BASELINE")
    print(f"{"="*50}")
    t0 = time.time()
    results[ct] = train_baseline(ct)
    elapsed = (time.time()-t0)/60
    print(f"  Completed in {elapsed:.1f}min")
    torch.save(results, "baseline_results.pt")
    print(f"  Checkpoint saved")

# Load GAT reference if available
gat_r2 = {}
try:
    df_ens = pd.read_csv("ensemble_results.csv")
    for _, row in df_ens.iterrows():
        gat_r2[row["Property"]] = row["Ensemble_R2"]
except:
    print("No ensemble_results.csv yet — will compare after CV completes")

print(f"\n{"="*70}")
print("BASELINE RESULTS")
print(f"{"="*70}")
header = f"{"Property":16s}  {"GCN R2":>8s}  {"GIN R2":>8s}"
if gat_r2:
    header += f"  {"GAT R2":>8s}"
print(header)
print("-" * (42 if not gat_r2 else 52))
for i, prop in enumerate(targets):
    line = f"{prop:16s}  {results["gcn"]["r2"][i]:8.4f}  {results["gin"]["r2"][i]:8.4f}"
    if gat_r2:
        line += f"  {gat_r2.get(prop, 0):8.4f}"
    print(line)

print(f"\nMean R2:")
print(f"  GCN: {results["gcn"]["mean_r2"]:.4f}")
print(f"  GIN: {results["gin"]["mean_r2"]:.4f}")
if gat_r2:
    print(f"  GAT: {np.mean(list(gat_r2.values())):.4f}")

with open("baseline_results.json", "w") as f:
    json.dump({k: {"r2": v["r2"], "mae": v["mae"], "mean_r2": v["mean_r2"]}
               for k,v in results.items()}, f, indent=2)
print("Saved to baseline_results.json")


Running GCN + GIN baselines (bs=512, torch.compile)

GCN BASELINE


  [GCN] epochs=200 bs=512 compile=True


W0801 14:32:10.548000 2858456 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] Graph break from `Tensor.item()`, consider setting:
W0801 14:32:10.548000 2858456 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0]     torch._dynamo.config.capture_scalar_outputs = True
W0801 14:32:10.548000 2858456 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] or:
W0801 14:32:10.548000 2858456 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W0801 14:32:10.548000 2858456 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] to include these operations in the captured graph.
W0801 14:32:10.548000 2858456 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] 
W0801 14:32:10.548000 2858456 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0] Graph break: from user code at:
W0801 14:32:10.548000 2858456 site-packages/torch/_dynamo/variables/tensor.py:1612] [0/0]   File "/run/media/SCR/ORAC/ipykernel_28

W0801 14:32:34.000000 2858456 site-packages/torch/_inductor/cudagraph_utils.py:434] [__cudagraphs] CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 9 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.


W0801 14:32:34.006000 2858456 site-packages/torch/_inductor/cudagraph_utils.py:434] [__cudagraphs] CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 9 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.


...

  [GIN] Ep 80: va=0.2593 pat=0


  [GIN] Ep 120: va=0.1799 pat=8


  [GIN] Ep 160: va=0.1809 pat=13


  [GIN] Ep 200: va=0.1571 pat=0


  [GIN] Mean R2=0.8420  gap=0.7345  dipole=0.5916
  Completed in 13.8min
  Checkpoint saved

BASELINE RESULTS
Property            GCN R2    GIN R2    GAT R2
----------------------------------------------------
gap                 0.6346    0.7345    0.8271
homo                0.7423    0.8901    0.9463
lumo                0.7402    0.8772    0.9364
dipole              0.3942    0.5916    0.6936
metal_charge        0.9004    0.9348    0.9680
E_elec              0.6900    0.9089    0.9654
E_disp              0.4901    0.8897    0.9226
polarizability      0.5260    0.9090    0.9349

Mean R2:
  GCN: 0.6397
  GIN: 0.8420
  GAT: 0.8993
Saved to baseline_results.json
